# 🔍 Tahap 2: Rangkaian Preprocessing Teks Langkah demi Langkah
**Proyek:** Sistem Rekomendasi & Analisis Sentimen Pariwisata Kabupaten Garut  
**Tujuan:**
Menampilkan secara transparan hasil dari setiap tahapan pemrosesan teks (*Text Preprocessing Pipeline*) dalam bentuk **Tabel Perbandingan Sebelum vs Sesudah (Before & After)**:
1. **Tahap 1: Pembersihan Teks Mentah (*Text Cleaning*)**: Menghapus URL, tag HTML, emoji, simbol, dan tanda baca.
2. **Tahap 2: Penyeragaman Huruf (*Case Folding*)**: Mengubah teks menjadi huruf kecil (*lowercase*).
3. **Tahap 3: Pemotongan Kata (*Tokenization*)**: Memecah kalimat menjadi kumpulan token kata.
4. **Tahap 4: Pembersihan Kata Tidak Bermakna (*Stopword Removal*)**: Menghapus stopword bahasa Indonesia (NLTK + custom domain Garut).
5. **Tahap 5: Pengembalian ke Kata Dasar (*Stemming*)**: Menggunakan PySastrawi untuk mereduksi kata berimbuhan.
6. **Tahap 6: Filter Ulasan Kosong (*Empty Review Filter*)**: Menyaring ulasan yang menjadi kosong setelah stopwords dibersihkan.


In [1]:
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')

import sys
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

CURRENT_DIR = Path.cwd()
ROOT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from collections import Counter

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

from config import settings
from config.constants import CUSTOM_INDONESIAN_STOPWORDS
from preprocessing.cleaning import clean_text
from preprocessing.case_folding import case_folding
from preprocessing.tokenization import tokenize
from preprocessing.stopword_removal import remove_stopwords, ALL_STOPWORDS
from preprocessing.stemming import stem_tokens

print(f"✅ Root Directory: {ROOT_DIR}")
print("Module preprocessing berhasil dimuat.")


✅ Root Directory: /Users/macbook/Garut-Tourism-Reccomendation
Module preprocessing berhasil dimuat.


## 1. Pemuatan Data Hasil Validasi Tahap 1

In [2]:
interim_path = settings.INTERIM_DATA_DIR / "reviews_validated.csv"
if not interim_path.exists():
    raise FileNotFoundError("reviews_validated.csv belum ada. Jalankan notebook 01 terlebih dahulu.")

df = pd.read_csv(interim_path)

# Tabel Ringkasan Input Preprocessing
input_summary = pd.DataFrame([
    {"Deskripsi": "Total Baris Ulasan Siap Preprocessing", "Nilai": f"{len(df):,} baris"},
    {"Deskripsi": "Jumlah Destinasi Wisata", "Nilai": f"{df['destination_name'].nunique():,} destinasi"},
    {"Deskripsi": "Rata-rata Rating Awal", "Nilai": f"{df['rating'].mean():.2f} / 5.0"}
])
display(input_summary)


,Deskripsi,Nilai
0,Total Baris Ulasan Siap Preprocessing,"18,124 baris"
1,Jumlah Destinasi Wisata,274 destinasi
2,Rata-rata Rating Awal,4.46 / 5.0


## 2. Tahap 1: Text Cleaning (Pembersihan Teks Mentah)
Fungsi `clean_text` membersihkan:
- URL (`http...`, `www...`)
- Tag HTML (`<...>` )
- Emoji dan karakter Non-ASCII
- Karakter khusus, simbol, dan tanda baca (diubah menjadi spasi)
- Spasi ganda, newline, dan tab (dinormalkan menjadi spasi tunggal)


In [3]:
df["step1_cleaned"] = df["review_text"].apply(clean_text)

# Hitung statistik panjang karakter
df["len_raw_char"] = df["review_text"].str.len()
df["len_clean_char"] = df["step1_cleaned"].str.len()

# TABEL STATISTIK TEXT CLEANING
cleaning_stat_table = pd.DataFrame([
    {"Metrik Karakter": "Rata-rata Panjang Karakter Sebelum Cleaning (Raw)", "Nilai": f"{df['len_raw_char'].mean():.1f} karakter"},
    {"Metrik Karakter": "Rata-rata Panjang Karakter Sesudah Cleaning", "Nilai": f"{df['len_clean_char'].mean():.1f} karakter"},
    {"Metrik Karakter": "Rata-rata Karakter Simbol/Emoji/Punctuation yang Terhapus", "Nilai": f"{df['len_raw_char'].mean() - df['len_clean_char'].mean():.1f} karakter"},
    {"Metrik Karakter": "Maksimum Panjang Karakter", "Nilai": f"{df['len_raw_char'].max():,} $\rightarrow$ {df['len_clean_char'].max():,} karakter"}
])
display(cleaning_stat_table)


,Metrik Karakter,Nilai
0,Rata-rata Panjang Karakter Sebelum Cleaning (Raw),138.1 karakter
1,Rata-rata Panjang Karakter Sesudah Cleaning,133.9 karakter
2,Rata-rata Karakter Simbol/Emoji/Punctuation yang Terhapus,4.2 karakter
3,Maksimum Panjang Karakter,"3,513 $\rightarrow$ 3,428 karakter"


In [4]:
# TABEL CONTOH PERBANDINGAN SEBELUM VS SESUDAH TEXT CLEANING
# Pilih 5 sampel ulasan riil yang mengandung tanda baca, emoji, link, atau karakter khusus
samples_symbols = df[df["review_text"].str.contains(r"[^\w\s]|http", regex=True, na=False)].sample(5, random_state=42)

tabel_step1 = pd.DataFrame({
    "No": range(1, 6),
    "Teks Mentah Awal (Sebelum Cleaning)": samples_symbols["review_text"].values,
    "Hasil Teks (Sesudah Cleaning)": samples_symbols["step1_cleaned"].values
})
display(tabel_step1)


,No,Teks Mentah Awal (Sebelum Cleaning),Hasil Teks (Sesudah Cleaning)
0,1,"Makanan nya beneran enak, untuk ayam kampung harganya wajar Tp kudu siapkan hati kl lagi rame, krn proses nya cukup membutuhkan waktu Sabar saja, lalu nikmati hidangannya jika sdh di meja Worth the wait",Makanan nya beneran enak untuk ayam kampung harganya wajar Tp kudu siapkan hati kl lagi rame krn proses nya cukup membutuhkan waktu Sabar saja lalu nikmati hidangannya jika sdh di meja Worth the wait
1,2,"Penginapan ini memiliki suasana yang sangat menyenangkan dengan nuansa alam yang menenangkan. Tersedia kolam renang air hangat untuk anak-anak maupun dewasa, sehingga nyaman untuk seluruh keluarga. Makanan yang disajikan enak dan pelayanan …",Penginapan ini memiliki suasana yang sangat menyenangkan dengan nuansa alam yang menenangkan Tersedia kolam renang air hangat untuk anak anak maupun dewasa sehingga nyaman untuk seluruh keluarga Makanan yang disajikan enak dan pelayanan
2,3,"Tidak menyangka. Food presentasinya juara. Rasa aman banget, anak2 suka. Ramah keluarga, kopinya oke!",Tidak menyangka Food presentasinya juara Rasa aman banget anak2 suka Ramah keluarga kopinya oke
3,4,"Makanannya mahal banget, roti 1 potongan kecil 15 rb, belum sama pajak. Tidak sesuai dengan harga di menu. Ada anak yg makan di kolam, makanannya dilempar ke kolam, dibiarkan gitu aja, begitu juga yg buang sampah plastik ke kolam.",Makanannya mahal banget roti 1 potongan kecil 15 rb belum sama pajak Tidak sesuai dengan harga di menu Ada anak yg makan di kolam makanannya dilempar ke kolam dibiarkan gitu aja begitu juga yg buang sampah plastik ke kolam
4,5,"Tempatnya bagus, pemandangan nya bagus, luas, ada Playground, tp makannya rasanya kurang enak, spageti aglio aneh rasanya, kalo gabisa masak spageti yauda jangan dijual atuh, ayam bakarnya bumbunya gak meresap, yg mendingan rasanya sayur asem dan ayam garang asam",Tempatnya bagus pemandangan nya bagus luas ada Playground tp makannya rasanya kurang enak spageti aglio aneh rasanya kalo gabisa masak spageti yauda jangan dijual atuh ayam bakarnya bumbunya gak meresap yg mendingan rasanya sayur asem dan ayam garang asam


## 3. Tahap 2: Case Folding (Penyeragaman Huruf Kecil)
Mengubah semua huruf kapital menjadi huruf kecil (*lowercase*) agar representasi kata seragam (*Garut*, *GARUT*, *garut* $ightarrow$ *garut*).


In [5]:
df["step2_folded"] = df["step1_cleaned"].apply(case_folding)

# TABEL CONTOH SEBELUM VS SESUDAH CASE FOLDING
samples_cf = df[df["step1_cleaned"].str.contains(r"[A-Z]", regex=True, na=False)].sample(5, random_state=42)

tabel_step2 = pd.DataFrame({
    "No": range(1, 6),
    "Sebelum Case Folding (Ada Huruf Kapital)": samples_cf["step1_cleaned"].values,
    "Sesudah Case Folding (Seluruhnya Huruf Kecil)": samples_cf["step2_folded"].values
})
display(tabel_step2)


,No,Sebelum Case Folding (Ada Huruf Kapital),Sesudah Case Folding (Seluruhnya Huruf Kecil)
0,1,Enak,enak
1,2,Bersih banget,bersih banget
2,3,Sebenarnya asyik banget seru sepanjang perjalanan dari tempat parkir ke Batu Lempar indah Udara sejuk Terbayar capek dengan pemandangan Tempat kemping yang sebenarnya kalau dikelola lebih baik akan lebih nyaman Mushola ada warung ada saung2 juga ada kamar mandi dengan air yang banyak pun tersedia Bakal lebih nyaman kalau lebih bersih dan fasilitas yang terpelihara Tapi tempatnya mah asyik banget,sebenarnya asyik banget seru sepanjang perjalanan dari tempat parkir ke batu lempar indah udara sejuk terbayar capek dengan pemandangan tempat kemping yang sebenarnya kalau dikelola lebih baik akan lebih nyaman mushola ada warung ada saung2 juga ada kamar mandi dengan air yang banyak pun tersedia bakal lebih nyaman kalau lebih bersih dan fasilitas yang terpelihara tapi tempatnya mah asyik banget
3,4,Suasana makanan dan pelayanan sangat top Terima kasih Joglo Abah Resto,suasana makanan dan pelayanan sangat top terima kasih joglo abah resto
4,5,Tempat nya lumayan enak buat berwisata,tempat nya lumayan enak buat berwisata


## 4. Tahap 3: Tokenization (Pemotongan Kata)
Memecah kalimat teks menjadi daftar token kata (*list of tokens*) berdasarkan spasi.


In [6]:
df["step3_tokens"] = df["step2_folded"].apply(tokenize)
df["num_tokens_raw"] = df["step3_tokens"].apply(len)

# TABEL STATISTIK TOKENISASI
token_stat_table = pd.DataFrame([
    {"Metrik": "Total Keseluruhan Kata (Tokens) di Dataset", "Nilai": f"{df['num_tokens_raw'].sum():,} kata"},
    {"Metrik": "Rata-rata Jumlah Kata per Ulasan", "Nilai": f"{df['num_tokens_raw'].mean():.2f} kata"},
    {"Metrik": "Jumlah Kata Terbanyak dalam 1 Ulasan", "Nilai": f"{df['num_tokens_raw'].max():,} kata"},
    {"Metrik": "Jumlah Kata Tersedikit dalam 1 Ulasan", "Nilai": f"{df['num_tokens_raw'].min():,} kata"}
])
display(token_stat_table)


,Metrik,Nilai
0,Total Keseluruhan Kata (Tokens) di Dataset,"389,858 kata"
1,Rata-rata Jumlah Kata per Ulasan,21.51 kata
2,Jumlah Kata Terbanyak dalam 1 Ulasan,510 kata
3,Jumlah Kata Tersedikit dalam 1 Ulasan,0 kata


In [7]:
# TABEL CONTOH SEBELUM VS SESUDAH TOKENIZATION
samples_tok = df.sample(5, random_state=42)

tabel_step3 = pd.DataFrame({
    "No": range(1, 6),
    "Kalimat Teks Input (String)": samples_tok["step2_folded"].values,
    "Hasil Tokenisasi (List of Words)": [str(toks) for toks in samples_tok["step3_tokens"].values],
    "Jumlah Token": samples_tok["num_tokens_raw"].values
})
display(tabel_step3)


,No,Kalimat Teks Input (String),Hasil Tokenisasi (List of Words),Jumlah Token
0,1,pantai yang indah dengan hamparan pasir membentang,"['pantai', 'yang', 'indah', 'dengan', 'hamparan', 'pasir', 'membentang']",7
1,2,semua sajian yang kami coba terasa enak hanya terganggu dengan banyaknya serangga,"['semua', 'sajian', 'yang', 'kami', 'coba', 'terasa', 'enak', 'hanya', 'terganggu', 'dengan', 'banyaknya', 'serangga']",12
2,3,jajanan beragam cuma parkir agak susah suka full,"['jajanan', 'beragam', 'cuma', 'parkir', 'agak', 'susah', 'suka', 'full']",8
3,4,makanan enakk tempat nyaman ada play ground anaknya,"['makanan', 'enakk', 'tempat', 'nyaman', 'ada', 'play', 'ground', 'anaknya']",8
4,5,ada mie baso enak namanya mie baso mitra ada digang sebelum tugu intan sebelah kanan,"['ada', 'mie', 'baso', 'enak', 'namanya', 'mie', 'baso', 'mitra', 'ada', 'digang', 'sebelum', 'tugu', 'intan', 'sebelah', 'kanan']",15


## 5. Tahap 4: Stopword Removal (Pembersihan Kata Tidak Bermakna)

Stopwords yang dieliminasi:
1. **NLTK Indonesian Stopwords**: Kata hubung (*dan, yang, di, ke, dari, adalah*).
2. **Custom Indonesian Stopwords**: Kata slang & pengisi ulasan (*yg, gak, tempat, wisata, garut, lokasi, nya, banget, deh*).


In [8]:
df["step4_no_stopwords"] = df["step3_tokens"].apply(remove_stopwords)
df["num_tokens_clean"] = df["step4_no_stopwords"].apply(len)

total_tokens_before = df["num_tokens_raw"].sum()
total_tokens_after = df["num_tokens_clean"].sum()
removed_tokens = total_tokens_before - total_tokens_after

# TABEL REKAPITULASI STOPWORD REMOVAL
sw_summary_table = pd.DataFrame([
    {"Kategori": "Total Kata Sebelum Stopword Removal", "Jumlah": f"{total_tokens_before:,} kata", "Persentase": "100.0%"},
    {"Kategori": "Total Kata Stopword yang Dihapus", "Jumlah": f"{removed_tokens:,} kata", "Persentase": f"{(removed_tokens/total_tokens_before)*100:.2f}%"},
    {"Kategori": "Total Kata Bermakna yang Dipertahankan", "Jumlah": f"{total_tokens_after:,} kata", "Persentase": f"{(total_tokens_after/total_tokens_before)*100:.2f}%"}
])
display(sw_summary_table)


,Kategori,Jumlah,Persentase
0,Total Kata Sebelum Stopword Removal,"389,858 kata",100.0%
1,Total Kata Stopword yang Dihapus,"147,052 kata",37.72%
2,Total Kata Bermakna yang Dipertahankan,"242,806 kata",62.28%


In [9]:
# TABEL TOP 20 STOPWORDS YANG PALING BANYAK DIHAPUS DARI ULASAN
all_raw_tokens = [tok for sublist in df["step3_tokens"] for tok in sublist]
removed_stopword_list = [tok for tok in all_raw_tokens if tok in ALL_STOPWORDS]
stopword_counter = Counter(removed_stopword_list).most_common(20)

df_top_sw = pd.DataFrame(stopword_counter, columns=["Kata Stopword", "Frekuensi Terhapus"])
df_top_sw["Persentase dari Seluruh Stopword"] = [(v / len(removed_stopword_list)) * 100 for v in df_top_sw["Frekuensi Terhapus"]]
df_top_sw["Persentase dari Seluruh Stopword"] = df_top_sw["Persentase dari Seluruh Stopword"].map("{:.2f}%".format)
df_top_sw.index = range(1, 21)
display(df_top_sw)


,Kata Stopword,Frekuensi Terhapus,Persentase dari Seluruh Stopword
1,dan,9419,6.41%
2,di,6670,4.54%
3,yang,5871,3.99%
4,ada,4501,3.06%
5,tempat,4148,2.82%
6,untuk,3931,2.67%
7,yg,3615,2.46%
8,juga,3518,2.39%
9,ke,2882,1.96%
10,banyak,2696,1.83%


In [10]:
# TABEL CONTOH SEBELUM VS SESUDAH STOPWORD REMOVAL
samples_sw = df.sample(5, random_state=42)

tabel_step4 = pd.DataFrame({
    "No": range(1, 6),
    "Token Sebelum Stopwords": [str(t) for t in samples_sw["step3_tokens"].values],
    "Token Sesudah Stopwords Dihapus": [str(t) for t in samples_sw["step4_no_stopwords"].values],
    "Jumlah Kata Dihapus": samples_sw["num_tokens_raw"].values - samples_sw["num_tokens_clean"].values
})
display(tabel_step4)


,No,Token Sebelum Stopwords,Token Sesudah Stopwords Dihapus,Jumlah Kata Dihapus
0,1,"['pantai', 'yang', 'indah', 'dengan', 'hamparan', 'pasir', 'membentang']","['pantai', 'indah', 'hamparan', 'pasir', 'membentang']",2
1,2,"['semua', 'sajian', 'yang', 'kami', 'coba', 'terasa', 'enak', 'hanya', 'terganggu', 'dengan', 'banyaknya', 'serangga']","['sajian', 'coba', 'enak', 'terganggu', 'banyaknya', 'serangga']",6
2,3,"['jajanan', 'beragam', 'cuma', 'parkir', 'agak', 'susah', 'suka', 'full']","['jajanan', 'beragam', 'parkir', 'susah', 'suka', 'full']",2
3,4,"['makanan', 'enakk', 'tempat', 'nyaman', 'ada', 'play', 'ground', 'anaknya']","['makanan', 'enakk', 'nyaman', 'play', 'ground', 'anaknya']",2
4,5,"['ada', 'mie', 'baso', 'enak', 'namanya', 'mie', 'baso', 'mitra', 'ada', 'digang', 'sebelum', 'tugu', 'intan', 'sebelah', 'kanan']","['mie', 'baso', 'enak', 'namanya', 'mie', 'baso', 'mitra', 'digang', 'tugu', 'intan', 'sebelah', 'kanan']",3


## 6. Tahap 5: Stemming (Pengubahan ke Kata Dasar)
Menggunakan **PySastrawi** untuk mereduksi kata berimbuhan bahasa Indonesia menjadi bentuk dasarnya (*root words*).


In [11]:
# Eksekusi Stemming dengan progress bar
tqdm.pandas(desc="Menjalankan Stemming")
df["step5_stemmed"] = df["step4_no_stopwords"].progress_apply(stem_tokens)

# Gabungkan token menjadi kalimat string bersih akhir
df["cleaned_text"] = df["step5_stemmed"].apply(lambda tokens: " ".join(tokens))


Menjalankan Stemming:   0%|          | 0/18124 [00:00<?, ?it/s]

Menjalankan Stemming:   2%|▏         | 328/18124 [00:00<00:05, 3279.19it/s]

Menjalankan Stemming:   4%|▎         | 656/18124 [00:00<00:05, 3171.90it/s]

Menjalankan Stemming:   5%|▌         | 974/18124 [00:00<00:08, 2131.79it/s]

Menjalankan Stemming:   8%|▊         | 1469/18124 [00:00<00:05, 2978.67it/s]

...

Menjalankan Stemming:  74%|███████▎  | 13359/18124 [00:01<00:00, 12204.10it/s]

Menjalankan Stemming:  83%|████████▎ | 15066/18124 [00:01<00:00, 13604.06it/s]

Menjalankan Stemming:  91%|█████████ | 16444/18124 [00:01<00:00, 13651.47it/s]

Menjalankan Stemming:  98%|█████████▊| 17822/18124 [00:01<00:00, 10986.97it/s]

Menjalankan Stemming: 100%|██████████| 18124/18124 [00:01<00:00, 9305.64it/s] 

In [12]:
# TABEL CONTOH PEMETAAN KATA BERIMBUHAN -> KATA DASAR (STEMMED)
sample_pairs = []
seen_pairs = set()
for orig_toks, stem_toks in zip(df["step4_no_stopwords"], df["step5_stemmed"]):
    for o, s in zip(orig_toks, stem_toks):
        if o != s and (o, s) not in seen_pairs:
            seen_pairs.add((o, s))
            sample_pairs.append({"Kata Berimbuhan (Sebelum)": o, "Kata Dasar / Root Word (Sesudah)": s})
            if len(sample_pairs) >= 15:
                break
    if len(sample_pairs) >= 15:
        break

df_stem_samples = pd.DataFrame(sample_pairs)
df_stem_samples.index = range(1, len(df_stem_samples) + 1)
display(df_stem_samples)


,Kata Berimbuhan (Sebelum),Kata Dasar / Root Word (Sesudah)
1,pemandangan,pandang
2,parkiran,parkir
3,pemula,mula
4,menggigil,gigil
5,mendaftar,daftar
6,pengalaman,alam
7,menyenangkan,senang
8,jalannya,jalan
9,minusnya,minus
10,membayar,bayar


## 7. Tahap 6: Evaluasi Akhir & Filter Ulasan Kosong
Setelah pembersihan stopwords dan stemming, beberapa ulasan yang aslinya hanya berisi stopwords menghasilkan string kosong (`""`). Baris kosong ini disaring agar model ML menerima data yang valid.


In [13]:
count_before_empty_filter = len(df)
df_final = df[df["cleaned_text"].str.strip() != ""].copy()
count_after_empty_filter = len(df_final)
empty_filtered_count = count_before_empty_filter - count_after_empty_filter

# TABEL CORONG DATA PREPROCESSING (DATA FUNNEL REKAP)
rekap_preprocessing = pd.DataFrame([
    {"Tahap Preprocessing": "1. Ulasan Masuk Tahap Preprocessing", "Jumlah Baris": f"{count_before_empty_filter:,}", "Ulasan Terfilter": "0", "Persentase": "100.0%"},
    {"Tahap Preprocessing": "2. Ulasan Menjadi Kosong Pasca Stopword Removal", "Jumlah Baris": f"-{empty_filtered_count:,}", "Ulasan Terfilter": f"-{empty_filtered_count:,}", "Persentase": f"{(empty_filtered_count/count_before_empty_filter)*100:.2f}%"},
    {"Tahap Preprocessing": "3. Ulasan BERSIH AKHIR (Siap Latih Model ML)", "Jumlah Baris": f"{count_after_empty_filter:,}", "Ulasan Terfilter": "0", "Persentase": f"{(count_after_empty_filter/count_before_empty_filter)*100:.2f}%"}
])
display(rekap_preprocessing)


,Tahap Preprocessing,Jumlah Baris,Ulasan Terfilter,Persentase
0,1. Ulasan Masuk Tahap Preprocessing,"18,124",0,100.0%
1,2. Ulasan Menjadi Kosong Pasca Stopword Removal,-201,-201,1.11%
2,3. Ulasan BERSIH AKHIR (Siap Latih Model ML),"17,923",0,98.89%


In [14]:
# TABEL PERBANDINGAN KOMPREHENSIF DARI RAW SAMPAI FINAL
sample_all_steps = df_final.sample(5, random_state=42)

tabel_komprehensif = pd.DataFrame({
    "No": range(1, 6),
    "1. Raw Review (Mentah)": sample_all_steps["review_text"].values,
    "2. Cleaned (Bebas Simbol/URL/Emoji)": sample_all_steps["step1_cleaned"].values,
    "3. Case Folded (Huruf Kecil)": sample_all_steps["step2_folded"].values,
    "4. Cleaned & Stemmed (Final Siap ML)": sample_all_steps["cleaned_text"].values
})
display(tabel_komprehensif)


,No,1. Raw Review (Mentah),2. Cleaned (Bebas Simbol/URL/Emoji),3. Case Folded (Huruf Kecil),4. Cleaned & Stemmed (Final Siap ML)
0,1,Bagus,Bagus,bagus,bagus
1,2,Nyaman buat kemping,Nyaman buat kemping,nyaman buat kemping,nyaman kemping
2,3,Tempat nyaman utk beristirahat,Tempat nyaman utk beristirahat,tempat nyaman utk beristirahat,nyaman istirahat
3,4,"Kampung Muara Sunda Garut menawarkan suasana yang sangat aesthetic dan sejuk, cocok sekali untuk makan bersama keluarga sambil menikmati udara asri. Rasa makanannya pun cukup enak dengan bumbu Sunda yang pas. Namun, jika kondisi restoran sedang penuh, pelayanannya terasa cukup lama, jadi disarankan datang lebih awal atau saat tidak sedang terburu-buru agar tetap nyaman..",Kampung Muara Sunda Garut menawarkan suasana yang sangat aesthetic dan sejuk cocok sekali untuk makan bersama keluarga sambil menikmati udara asri Rasa makanannya pun cukup enak dengan bumbu Sunda yang pas Namun jika kondisi restoran sedang penuh pelayanannya terasa cukup lama jadi disarankan datang lebih awal atau saat tidak sedang terburu buru agar tetap nyaman,kampung muara sunda garut menawarkan suasana yang sangat aesthetic dan sejuk cocok sekali untuk makan bersama keluarga sambil menikmati udara asri rasa makanannya pun cukup enak dengan bumbu sunda yang pas namun jika kondisi restoran sedang penuh pelayanannya terasa cukup lama jadi disarankan datang lebih awal atau saat tidak sedang terburu buru agar tetap nyaman,kampung muara sunda tawar suasana aesthetic sejuk cocok makan keluarga nikmat udara asri makan enak bumbu sunda pas kondisi restoran penuh layan saran buru buru nyaman
4,5,"Saya datang ke sini untuk sarapan sekitar jam 8 pagi dan jujur ​​saja, saya sama sekali tidak kecewa. Makanannya luar biasa, favorit saya adalah roti bakar, benar-benar lezat. Pemandangannya membuat sarapan saya semakin sempurna. Untungnya hari itu langit lebih cerah dari biasanya. Pengalaman 10/10.",Saya datang ke sini untuk sarapan sekitar jam 8 pagi dan jujur saja saya sama sekali tidak kecewa Makanannya luar biasa favorit saya adalah roti bakar benar benar lezat Pemandangannya membuat sarapan saya semakin sempurna Untungnya hari itu langit lebih cerah dari biasanya Pengalaman 10 10,saya datang ke sini untuk sarapan sekitar jam 8 pagi dan jujur saja saya sama sekali tidak kecewa makanannya luar biasa favorit saya adalah roti bakar benar benar lezat pemandangannya membuat sarapan saya semakin sempurna untungnya hari itu langit lebih cerah dari biasanya pengalaman 10 10,sarap jam 8 pagi jujur kecewa makan favorit roti bakar lezat pandang sarap sempurna untung langit cerah alam 10 10


In [15]:
# Simpan dataset hasil preprocessing final
output_final_path = settings.FINAL_DATA_DIR / "processed_reviews.csv"
output_cols = ["destination_name", "author", "rating", "review_text", "cleaned_text"]
df_final[output_cols].to_csv(output_final_path, index=False)
print(f"✅ Data ulasan bersih final berhasil disimpan ke: {output_final_path}")
print(f"Total baris ulasan bersih final: {len(df_final):,} baris")


✅ Data ulasan bersih final berhasil disimpan ke: /Users/macbook/Garut-Tourism-Reccomendation/data/final/processed_reviews.csv
Total baris ulasan bersih final: 17,923 baris
